# synthkit worked example: Titanic

Chosen for what Adult Census doesn't exercise: null co-occurrence (`Cabin` is null ~77% of
the time), a free-text-ish `Name` column with no repeats, a messy `Ticket` identifier column,
and a much smaller dataset (891 rows) relative to Adult Census's 32,561.

Testing against this dataset during development surfaced two real bugs, a privacy leak on
small all-unique-string columns, and a combinatorial blowup in the rare-combination privacy
check, both fixed.

In [1]:
import subprocess
from pathlib import Path

import pandas as pd

import synthkit as sk

DATA_DIR = Path("../data")
DATA_PATH = DATA_DIR / "titanic.csv"
DATA_URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

DATA_DIR.mkdir(parents=True, exist_ok=True)
if not DATA_PATH.exists():
    subprocess.run(["curl", "-sL", "-o", str(DATA_PATH), DATA_URL], check=True)

df = pd.read_csv(DATA_PATH).drop(columns=["PassengerId"])
df.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
profile = sk.fit(df)
for column, ctype in profile.column_types.items():
    print(f"{column:12s} {ctype}")

Survived     categorical
Pclass       categorical
Name         identifier
Sex          categorical
Age          continuous
SibSp        categorical
Parch        categorical
Ticket       text
Fare         continuous
Cabin        text
Embarked     categorical


## Null co-occurrence: does the null rate survive the round trip?

In [3]:
synthetic = sk.emit(profile, n=len(df), seed=0)

for column in ["Age", "Cabin", "Embarked"]:
    real_rate = df[column].isna().mean()
    synth_rate = synthetic[column].isna().mean()
    print(f"{column:10s} real={real_rate:.3f} synthetic={synth_rate:.3f}")

Age        real=0.199 synthetic=0.199
Cabin      real=0.771 synthetic=0.776
Embarked   real=0.002 synthetic=0.000


## Category frequencies

In [4]:
for column in ["Sex", "Pclass", "Embarked"]:
    print(column)
    print("  real:     ", df[column].value_counts(normalize=True).round(3).to_dict())
    print("  synthetic:", synthetic[column].value_counts(normalize=True).round(3).to_dict())

Sex
  real:      {'male': 0.648, 'female': 0.352}
  synthetic: {'male': 0.671, 'female': 0.329}
Pclass
  real:      {3: 0.551, 1: 0.242, 2: 0.207}
  synthetic: {3: 0.521, 1: 0.248, 2: 0.231}
Embarked
  real:      {'S': 0.724, 'C': 0.189, 'Q': 0.087}
  synthetic: {'S': 0.743, 'C': 0.178, 'Q': 0.079}


## Names never leak

`Name` is all-unique, so it's classified an identifier and regenerated from a format pattern
-- never modeled statistically. No real passenger name should appear in the output.

In [5]:
assert profile.column_types["Name"] == "identifier"
leaked = set(synthetic["Name"]) & set(df["Name"])
print(f"real names appearing in synthetic output: {len(leaked)}")

real names appearing in synthetic output: 0


## Privacy check

In [6]:
report = sk.check(synthetic, profile, real=df, min_dcr_ratio=0.5)
print(f"dcr_ratio: {report.dcr_ratio:.3f}")
print(f"exact_matches: {report.exact_matches}")
print(f"rare_combination_leaks: {report.rare_combination_leaks}")
print(
    "A nonzero rare-combination count is expected on a small, several-categorical-column "
    "dataset like this one."
)

dcr_ratio: 1.431
exact_matches: 0
rare_combination_leaks: 86
A nonzero rare-combination count is expected on a small, several-categorical-column dataset like this one.
